# Merging `cleaned_sheet_cpi.csv` (base 2012 series) with `cpi_1664.xlsx` (base 2024 series)

This notebook standardizes column names/state names in the new Excel series so it matches
the schema of `cleaned_sheet_cpi.csv`, then concatenates the two into one combined long-format table.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Step 1: Load both files
Update the paths below to point to your local copies.

In [2]:
# Old / base dataset (already cleaned, base year 2012)
df_old = pd.read_csv(r"C:\Users\himan\Education1\Projects\MoSPI_CPI\Data\merged\cleaned_sheet_cpi_12-24.csv")


df_new = pd.read_excel(r"C:\Users\himan\Education1\Projects\MoSPI_CPI\Data\merged\cpi_aggregated_25-26.xlsx")

In [3]:
df_new.head()

,base_year,series,year,month,state,sector,parent_group,subgroup,index,inflation
0,2024,Current,2025,April,All India,Combined,Clothing and Footwear,Clothing,103.89,NaN
1,2024,Current,2025,April,All India,Combined,Clothing and Footwear,Footwear,103.30,NaN
2,2024,Current,2025,April,All India,Combined,Food and Beverages,Cereals and Products,104.13,NaN
3,2024,Current,2025,April,All India,Combined,Food and Beverages,Egg,101.61,NaN
4,2024,Current,2025,April,All India,Combined,Food and Beverages,Fruits,112.92,NaN


In [4]:
df_new=df_new.drop(columns=["series"])


In [5]:
df_old[df_old["Year_i"] == 2014].head(15)

,Unnamed: 0,BYear,Year_i,Month_i,State_i,Sector,Group_i,SubGroup,Index_i,Inflation
195585,195585,2012,2014,December,All India,Combined,Food and Beverages,Cereals and Products,122,3.19
195586,195586,2012,2014,December,All India,Combined,Food and Beverages,Meat and Fish,123,5.57
195587,195587,2012,2014,December,All India,Combined,Food and Beverages,Egg,123,0.00
195588,195588,2012,2014,December,All India,Combined,Food and Beverages,Fruits,125,10.94
195589,195589,2012,2014,December,All India,Combined,Food and Beverages,Vegetables,140,-3.37
195590,195590,2012,2014,December,All India,Combined,Food and Beverages,Pulses and Products,117,8.19
195591,195591,2012,2014,December,All India,Combined,Food and Beverages,Spices,119,7.78
195592,195592,2012,2014,December,All India,Combined,Food and Beverages,"Prepared Meals, Snacks, Sweets etc.",126,7.41
195593,195593,2012,2014,December,All India,Combined,Food and Beverages,Non-alcoholic Beverages,116,4.19
195594,195594,2012,2014,December,All India,Combined,Food and Beverages,Residual,123,4.39


In [6]:
df_new.head()

,base_year,year,month,state,sector,parent_group,subgroup,index,inflation
0,2024,2025,April,All India,Combined,Clothing and Footwear,Clothing,103.89,NaN
1,2024,2025,April,All India,Combined,Clothing and Footwear,Footwear,103.30,NaN
2,2024,2025,April,All India,Combined,Food and Beverages,Cereals and Products,104.13,NaN
3,2024,2025,April,All India,Combined,Food and Beverages,Egg,101.61,NaN
4,2024,2025,April,All India,Combined,Food and Beverages,Fruits,112.92,NaN


## Step 2: Align schemas

`df_new` uses MoSPI's new column names (`base_year`, `year`, `month`, `state`, `division`, `class`, ...).
We rename them to match `df_old`'s schema (`BYear`, `Year_i`, `Month_i`, `State_i`, `Group_i`, `SubGroup`, ...),
the same way the original cleaning notebook renamed `BaseYear -> BYear`, `Year -> Year_i`, etc.

In [7]:
df_new.rename(columns={
    "base_year": "BYear",
    "year": "Year_i",
    "month": "Month_i",
    "state": "State_i",
    "sector": "Sector",
    "parent_group": "Group_i",   # top-level category, plays the role of old "Group_i"
    "subgroup": "SubGroup",     # most granular category, plays the role of old "SubGroup"
    "index": "Index_i",
    "inflation": "Inflation",
}, inplace=True)

# Keep only the columns that exist in df_old, in the same order
common_cols = ["BYear", "Year_i", "Month_i", "State_i", "Sector", "Group_i", "SubGroup", "Index_i", "Inflation"]
df_new = df_new[common_cols]
df_new.head()

,BYear,Year_i,Month_i,State_i,Sector,Group_i,SubGroup,Index_i,Inflation
0,2024,2025,April,All India,Combined,Clothing and Footwear,Clothing,103.89,NaN
1,2024,2025,April,All India,Combined,Clothing and Footwear,Footwear,103.30,NaN
2,2024,2025,April,All India,Combined,Food and Beverages,Cereals and Products,104.13,NaN
3,2024,2025,April,All India,Combined,Food and Beverages,Egg,101.61,NaN
4,2024,2025,April,All India,Combined,Food and Beverages,Fruits,112.92,NaN


## Step 3: Fix state-name mismatches

The new series renamed/reorganized a few states & UTs. We map the ones that are really the
same place under a new label, so they line up with `df_old`. Genuinely new UTs
(`Ladakh`, the merged `Dadra and Nagar Haveli and Daman and Diu`) are kept as-is since they
have no equivalent row in the old series.

In [8]:
state_fix = {
    "Andaman And Nicobar Islands": "Andaman and Nicobar Islands",
    "Jammu And Kashmir": "Jammu and Kashmir",
    "NCT of Delhi": "Delhi",
}
df_new["State_i"] = df_new["State_i"].replace(state_fix)

df_new.loc[df_new["State_i"] == "Delhi"]  # sanity check

,BYear,Year_i,Month_i,State_i,Sector,Group_i,SubGroup,Index_i,Inflation
1292,2024,2025,April,Delhi,Combined,Clothing and Footwear,Clothing,104.43,NaN
1293,2024,2025,April,Delhi,Combined,Clothing and Footwear,Footwear,102.60,NaN
1294,2024,2025,April,Delhi,Combined,Food and Beverages,Cereals and Products,103.94,NaN
1295,2024,2025,April,Delhi,Combined,Food and Beverages,Egg,101.66,NaN
1296,2024,2025,April,Delhi,Combined,Food and Beverages,Fruits,108.32,NaN
...,...,...,...,...,...,...,...,...,...
38964,2024,2026,May,Delhi,Urban,Miscellaneous,Household Goods and Services,103.32,0.69
38965,2024,2026,May,Delhi,Urban,Miscellaneous,"Prepared Meals, Snacks, Sweets etc.",104.86,3.02
38966,2024,2026,May,Delhi,Urban,Miscellaneous,Residual,110.22,5.66
38967,2024,2026,May,Delhi,Urban,Miscellaneous,Transport and Communication,101.87,0.87


## Step 4: Clean dtypes
`Index_i` should be an integer, same as in `df_old`.

In [9]:
df_new["Index_i"] = pd.to_numeric(df_new["Index_i"], errors="coerce").round().astype("Int64")
df_new.info()

<class 'pandas.DataFrame'>
RangeIndex: 39710 entries, 0 to 39709
Data columns (total 9 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   BYear      39710 non-null  int64  
 1   Year_i     39710 non-null  int64  
 2   Month_i    39710 non-null  str    
 3   State_i    39710 non-null  str    
 4   Sector     39710 non-null  str    
 5   Group_i    39710 non-null  str    
 6   SubGroup   39710 non-null  str    
 7   Index_i    39710 non-null  Int64  
 8   Inflation  14630 non-null  float64
dtypes: Int64(1), float64(1), int64(2), str(5)
memory usage: 2.8 MB


## Step 5: Merge (concatenate) the two datasets

The two files cover **different base years / time periods** (2012 base -> 2014-2024,
2024 base -> 2025-2026) and don't share a common key to join on row-by-row, so the
correct "merge" here is a **vertical concat**: stack the new rows underneath the old
ones to extend the time series. `BYear` lets you tell the two series apart later.

In [10]:
merged = pd.concat([df_old, df_new], ignore_index=True, sort=False)
merged.sort_values(["Year_i", "Month_i", "State_i"], inplace=True)
merged.reset_index(drop=True, inplace=True)

print("Old rows   :", df_old.shape[0])
print("New rows   :", df_new.shape[0])
print("Merged rows:", merged.shape[0])
merged.head()
merged.tail()

Old rows   : 215961
New rows   : 39710
Merged rows: 255671


,Unnamed: 0,BYear,Year_i,Month_i,State_i,Sector,Group_i,SubGroup,Index_i,Inflation
255666,NaN,2024,2026,May,West Bengal,Urban,Miscellaneous,Household Goods and Services,102,1.12
255667,NaN,2024,2026,May,West Bengal,Urban,Miscellaneous,"Prepared Meals, Snacks, Sweets etc.",107,3.09
255668,NaN,2024,2026,May,West Bengal,Urban,Miscellaneous,Residual,106,3.93
255669,NaN,2024,2026,May,West Bengal,Urban,Miscellaneous,Transport and Communication,104,3.36
255670,NaN,2024,2026,May,West Bengal,Urban,"Pan, Tobacco and Intoxicants",Residual,110,5.44


In [11]:
merged["BYear"].value_counts()

BYear
2012    215961
2024     39710
Name: count, dtype: int64

## Step 6: Rebase `Index_i` from 2024=100 to 2012=100

Formula (derived above from the MoSPI/PIB linking factor):

```
Index_i(2012-base) = Index_i(2024-base) / LF[Sector]
```

`Inflation` (the year-on-year % change) is **left unchanged** — multiplying/dividing an entire
index series by a constant factor does not change percentage changes within that series, so the
published `Inflation` figures remain valid after rebasing.

In [12]:
# Official MoSPI/PIB linking factors (12 Feb 2026 press release, general index level)
LINKING_FACTOR = {"Rural": 0.5222, "Urban": 0.5320, "Combined": 0.5267}

df_new["Index_i"] = pd.to_numeric(df_new["Index_i"], errors="coerce")
df_new["Index_i"] = df_new["Index_i"] / df_new["Sector"].map(LINKING_FACTOR)
df_new["Index_i"] = df_new["Index_i"].round().astype("Int64")

# All rows are now expressed on the 2012 base
df_new["BYear"] = 2012

df_new.head()

,BYear,Year_i,Month_i,State_i,Sector,Group_i,SubGroup,Index_i,Inflation
0,2012,2025,April,All India,Combined,Clothing and Footwear,Clothing,197,NaN
1,2012,2025,April,All India,Combined,Clothing and Footwear,Footwear,196,NaN
2,2012,2025,April,All India,Combined,Food and Beverages,Cereals and Products,197,NaN
3,2012,2025,April,All India,Combined,Food and Beverages,Egg,194,NaN
4,2012,2025,April,All India,Combined,Food and Beverages,Fruits,215,NaN


## Step 7: Merge (concatenate) the two — now base-consistent — datasets

Both series are now on **base 2012=100**, so it's safe to stack them into one continuous
time series (2014-2026).

In [13]:
merged = pd.concat([df_old, df_new], ignore_index=True, sort=False)
print("Old rows   :", df_old.shape[0])
print("New rows   :", df_new.shape[0])
print("Merged rows:", merged.shape[0])

Old rows   : 215961
New rows   : 39710
Merged rows: 255671


## Step 8: Arrange dataset — sort by Year, then calendar Month order

By default, sorting `Month_i` as plain text sorts alphabetically (April, August, December, ...).
We instead make it a **categorical column with an explicit Jan -> Dec order** so the sort follows
the calendar.

In [14]:
month_order = ["January", "February", "March", "April", "May", "June",
               "July", "August", "September", "October", "November", "December"]

merged["Month_i"] = pd.Categorical(merged["Month_i"], categories=month_order, ordered=True)

merged.sort_values(["Year_i", "Month_i", "State_i"], inplace=True)
merged.reset_index(drop=True, inplace=True)

merged.head(15)

,Unnamed: 0,BYear,Year_i,Month_i,State_i,Sector,Group_i,SubGroup,Index_i,Inflation
0,214263.0,2012,2014,January,All India,Combined,Food and Beverages,Cereals and Products,119,10.33
1,214264.0,2012,2014,January,All India,Combined,Food and Beverages,Meat and Fish,118,10.72
2,214265.0,2012,2014,January,All India,Combined,Food and Beverages,Egg,124,12.82
3,214266.0,2012,2014,January,All India,Combined,Food and Beverages,Fruits,113,10.37
4,214267.0,2012,2014,January,All India,Combined,Food and Beverages,Vegetables,122,19.57
5,214268.0,2012,2014,January,All India,Combined,Food and Beverages,Pulses and Products,108,2.74
6,214269.0,2012,2014,January,All India,Combined,Food and Beverages,Spices,111,8.08
7,214270.0,2012,2014,January,All India,Combined,Food and Beverages,"Prepared Meals, Snacks, Sweets etc.",118,10.07
8,214271.0,2012,2014,January,All India,Combined,Food and Beverages,Non-alcoholic Beverages,112,7.05
9,214272.0,2012,2014,January,All India,Combined,Food and Beverages,Residual,115,9.66


In [15]:
merged = merged.drop(columns=["Unnamed: 0"])

In [16]:
merged.head()

,BYear,Year_i,Month_i,State_i,Sector,Group_i,SubGroup,Index_i,Inflation
0,2012,2014,January,All India,Combined,Food and Beverages,Cereals and Products,119,10.33
1,2012,2014,January,All India,Combined,Food and Beverages,Meat and Fish,118,10.72
2,2012,2014,January,All India,Combined,Food and Beverages,Egg,124,12.82
3,2012,2014,January,All India,Combined,Food and Beverages,Fruits,113,10.37
4,2012,2014,January,All India,Combined,Food and Beverages,Vegetables,122,19.57


In [17]:
merged.tail()

,BYear,Year_i,Month_i,State_i,Sector,Group_i,SubGroup,Index_i,Inflation
255666,2012,2026,July,West Bengal,Urban,Miscellaneous,Household Goods and Services,192,1.66
255667,2012,2026,July,West Bengal,Urban,Miscellaneous,"Prepared Meals, Snacks, Sweets etc.",209,7.02
255668,2012,2026,July,West Bengal,Urban,Miscellaneous,Residual,199,3.00
255669,2012,2026,July,West Bengal,Urban,Miscellaneous,Transport and Communication,195,3.38
255670,2012,2026,July,West Bengal,Urban,"Pan, Tobacco and Intoxicants",Residual,207,5.55


## Step 9: Save the merged, rebased, chronologically-sorted file

In [18]:
merged["Month_i"] = merged["Month_i"].astype(str)  # convert back from Categorical before saving
merged.to_csv(r"C:\Users\himan\Education1\Projects\MoSPI_CPI\Data\merged\merged_cpi_rebased_2012.csv", index=False)

In [19]:
merged["Year_i"].value_counts().loc[[2024, 2025]]

Year_i
2024    20376
2025    25080
Name: count, dtype: int64

In [20]:
a= merged.loc[merged["Year_i"] == 2024, "SubGroup"].unique()
a
print(a.value_counts)

<bound method StringArray.value_counts of <StringArray>
[               'Cereals and Products',                       'Meat and Fish',
                                 'Egg',                              'Fruits',
                          'Vegetables',                 'Pulses and Products',
                              'Spices', 'Prepared Meals, Snacks, Sweets etc.',
             'Non-alcoholic Beverages',                            'Residual',
                            'Clothing',                            'Footwear',
        'Household Goods and Services',                              'Health',
         'Transport and Communication',                           'Education']
Length: 16, dtype: str>


In [21]:
print(merged.loc[merged["Year_i"] == 2024, "Group_i"].unique())

<StringArray>
[          'Food and Beverages', 'Pan, Tobacco and Intoxicants',
        'Clothing and Footwear',                      'Housing',
               'Fuel and Light',                'Miscellaneous']
Length: 6, dtype: str


In [22]:
print(merged.loc[merged["Year_i"] == 2025, "SubGroup"].unique())

<StringArray>
[                           'Clothing',                            'Footwear',
                'Cereals and Products',                                 'Egg',
                              'Fruits',                       'Meat and Fish',
             'Non-alcoholic Beverages', 'Prepared Meals, Snacks, Sweets etc.',
                            'Residual',                          'Vegetables',
                             'Housing',                           'Education',
                              'Health',        'Household Goods and Services',
         'Transport and Communication']
Length: 15, dtype: str


In [23]:
print(merged.loc[merged["Year_i"] == 2025, "Group_i"].unique())

<StringArray>
[       'Clothing and Footwear',           'Food and Beverages',
                      'Housing',                'Miscellaneous',
 'Pan, Tobacco and Intoxicants']
Length: 5, dtype: str
